In [ ]:
import os
import io
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ee
from shapely import wkt

# --- 1. CONFIGURATION ET VISUALISATION ---
os.makedirs("output", exist_ok=True)
os.makedirs("data_project", exist_ok=True)

# Authentification GEE
print(" Connexion a Google Earth Engine...")
try:
    ee.Initialize(project='ici votre nom de projet')
    print(" Authentification reussie.")
except:
    ee.Authenticate()
    ee.Initialize(project='ici votre nom de projet')

# Configuration Visuelle (Classes NDVI)
NDVI_COLORS = ['#1f77b4', '#d3d3d3', '#ccff66', '#33cc33', '#006400']
NDVI_BOUNDS = [-1, 0, 0.2, 0.4, 0.6, 1]
cmap_ndvi = mcolors.ListedColormap(NDVI_COLORS)
norm_ndvi = mcolors.BoundaryNorm(NDVI_BOUNDS, cmap_ndvi.N)

# --- 2. CONFIGURATION DES VILLES ---
CITIES_CONFIG = [
    {
        "name": "Nantes",
        "url": "https://data.nantesmetropole.fr/api/explore/v2.1/catalog/datasets/244400404_quartiers-communes-nantes-metropole/exports/geojson",
        "ndvi_file": "data_project/Nantes_NDVI_2025.tif",
        "col_name": "nom"
    },
    {
        "name": "Rennes",
        "url": "https://data.rennesmetropole.fr/api/explore/v2.1/catalog/datasets/perimetres-des-12-quartiers-de-la-ville-de-rennes/exports/geojson",
        "ndvi_file": "data_project/Rennes_NDVI_2025.tif",
        "col_name": "nom"
    },
    {
        "name": "Angers",
        "url": "https://data.angers.fr/api/explore/v2.1/catalog/datasets/ccq_007/exports/geojson",
        "ndvi_file": "data_project/Angers_NDVI_2025.tif",
        "col_name": "libelle"
    },
    {
        "name": "Brest",
        "url": "https://geo.brest-metropole.fr/arcgis/rest/services/GPB_LIM_CC48/MapServer/1610008/query?where=1%3D1&outFields=*&f=geojson",
        "ndvi_file": "data_project/Brest_NDVI_2025.tif",
        "col_name": "LIBEL"
    }
]

In [2]:
def load_city_map(config):
    """Telecharge la carte, corrige la geometrie et standardise les noms de colonnes."""
    print(f"    Chargement de la carte : {config['name']}...")
    try:
        # Telechargement
        r = requests.get(config['url'], headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
        if r.status_code != 200: return None

        # Chargement
        try: gdf = gpd.read_file(io.BytesIO(r.content))
        except: gdf = gpd.read_file(io.BytesIO(r.content), driver='GeoJSON')

        # Correction de la projection et de la geometrie
        if not gdf.crs: gdf.set_crs(epsg=2154, inplace=True)
        gdf = gdf.to_crs(epsg=2154)

        # Aplatissement 3D vers 2D
        if gdf.geometry.has_z.any():
            gdf.geometry = gdf.geometry.map(lambda g: wkt.loads(wkt.dumps(g, output_dimension=2)))
        gdf.geometry = gdf.geometry.buffer(0)

        # Standardisation de la colonne des noms
        target = config.get('col_name')
        if target and target in gdf.columns:
            gdf["nom"] = gdf[target]
        else:
            # Solution de repli
            c = [x for x in gdf.columns if "nom" in x.lower() or "lib" in x.lower()]
            gdf["nom"] = gdf[c[0]] if c else config['name']

        return gdf
    except Exception as e:
        print(f"      Erreur Carte : {e}")
        return None

def download_satellite_image(gdf, output_path):
    """Telecharge l'image NDVI Sentinel-2 depuis Google Earth Engine."""
    if os.path.exists(output_path):
        print("      L'image existe deja.")
        return output_path

    print("     Requete donnees satellite (GEE)...")
    try:
        # 1. Conversion de la zone d'interet au format GEE
        roi = ee.Geometry.Rectangle(list(gdf.to_crs(4326).total_bounds))

        # 2. Collection Sentinel-2 (Ete 2024)
        s2 = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
              .filterBounds(roi)
              .filterDate('2024-06-01', '2024-09-30') # Ete 2024
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
              .median()
              .clip(roi))

        # 3. Calcul du NDVI
        ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')

        # 4. URL de telechargement
        url = ndvi.getDownloadURL({
            'scale': 10,
            'crs': 'EPSG:4326',
            'region': roi,
            'format': 'GEO_TIFF'
        })

        # 5. Sauvegarde sur le disque
        r = requests.get(url)
        with open(output_path, 'wb') as f:
            f.write(r.content)
        print("      Image telechargee.")
        return output_path
    except Exception as e:
        print(f"      Erreur GEE : {e}")
        return None

In [3]:
def analyze_vegetation(gdf, raster_path):
    """
    Calcul robuste du NDVI utilisant un masquage manuel.
    Previent les erreurs lorsque les geometries ne se chevauchent pas parfaitement.
    """
    print(f"    Analyse de la vegetation...")
    try:
        with rasterio.open(raster_path) as src:
            # Alignement des projections
            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)

            means = []

            # Boucle sur chaque quartier
            for _, row in gdf.iterrows():
                try:
                    # Decoupage de l'image selon la forme du quartier
                    out_image, _ = mask(src, [row.geometry], crop=True)
                    data = out_image[0]

                    # Filtrage des pixels valides
                    valid = data[(data != src.nodata) & (~np.isnan(data))]

                    if valid.size > 0:
                        means.append(np.mean(valid))
                    else:
                        means.append(None)
                except:
                    means.append(None) # Geometrie hors de l'image

            # Sauvegarde des statistiques
            gdf["ndvi_moyen"] = means

            # Nettoyage des resultats vides
            gdf_clean = gdf.dropna(subset=["ndvi_moyen"]).copy()

            if len(gdf_clean) > 0:
                top = gdf_clean.loc[gdf_clean["ndvi_moyen"].idxmax()]
                print(f"      Le plus vert : {top['nom']} ({top['ndvi_moyen']:.3f})")
                return gdf_clean

            return None

    except Exception as e:
        print(f"      Erreur d'analyse : {e}")
        return None

In [ ]:
print("--- DEMARRAGE DU PIPELINE ---")
final_results = []

for config in CITIES_CONFIG:
    print(f"\n TRAITEMENT : {config['name']}")

    # 1. Geometrie
    gdf = load_city_map(config)
    if gdf is None: continue

    # 2. Image Satellite
    tif = download_satellite_image(gdf, config['ndvi_file'])
    if not tif: continue

    # 3. Analyse
    res = analyze_vegetation(gdf, tif)

    if res is not None:
        res["city"] = config['name']
        final_results.append(res)
        # Sauvegarde du resultat par ville
        res.to_file(f"output/Result_{config['name']}.gpkg", driver="GPKG")

# --- SYNTHESE ---
if final_results:
    print("\n--- RAPPORT FINAL ---")
    df = pd.concat(final_results, ignore_index=True)

    # Top 5
    if "ndvi_moyen" in df.columns:
        top5 = df.sort_values("ndvi_moyen", ascending=False).head(5)
        print("\n TOP 5 DES QUARTIERS LES PLUS VERTS (GRAND OUEST) :")
        print(top5[["nom", "city", "ndvi_moyen"]].to_string(index=False))

    df.to_csv("output/Bilan_Complet.csv", index=False)
    print("\n TERMINE. Tous les fichiers sont dans le dossier 'output/'.")
else:
    print("\n ECHEC. Aucun resultat.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import pandas as pd
import geopandas as gpd
import os
import rasterio
from rasterio.plot import show
from rasterio.mask import mask
import numpy as np

print("--- DEMARRAGE DE LA GENERATION DES CARTES ---")

# ==========================================
# 1. CONFIGURATION : COULEURS ET CLASSES
# ==========================================

# Definition des classes et des couleurs
NDVI_LABELS = ["Eau / Artefact", "Bati / Sol nu", "Vegetation Basse", "Vegetation Moyenne", "Vegetation Dense"]
NDVI_COLORS_LIST = ['#1f77b4', '#d3d3d3', '#ccff66', '#33cc33', '#006400'] # Bleu, Gris, Jaune-Vert, Vert, Vert Fonce

# Dictionnaire pour la carte vectorielle (Carte 1)
CATEGORY_COLORS = {
    "Eau / Artefact": "#1f77b4",
    "Bati / Sol nu": "#d3d3d3",
    "Vegetation Basse": "#ccff66",
    "Vegetation Moyenne": "#33cc33",
    "Vegetation Dense": "#006400",
    "Inconnu": "#ffffff"
}

# Carte de couleurs pour la carte pixelisee (Carte 2)
NDVI_BOUNDS = [-1, 0, 0.2, 0.4, 0.6, 1]
# Initialisation de la carte de couleurs
cmap_base = mcolors.ListedColormap(NDVI_COLORS_LIST)
# Definir la couleur pour les valeurs masquees/incorrectes en blanc (fond transparent/blanc)
cmap_base.set_bad(color='white')
norm_ndvi = mcolors.BoundaryNorm(NDVI_BOUNDS, cmap_base.N)

def reclassify_ndvi(val):
    """Traduit le NDVI numerique en etiquettes textuelles."""
    if val is None or pd.isna(val): return "Inconnu"
    if val < 0: return "Eau / Artefact"
    if val < 0.2: return "Bati / Sol nu"
    if val < 0.4: return "Vegetation Basse"
    if val < 0.6: return "Vegetation Moyenne"
    return "Vegetation Dense"

# Chargement des donnees (Rechargement robuste)
if 'final_results' in locals() and final_results:
    full_gdf = pd.concat(final_results, ignore_index=True)
else:
    print("    Rechargement des donnees depuis le dossier output...")
    # Suppose que CITIES_CONFIG est defini dans les blocs precedents
    gpkgs = [f"output/Result_{c['name']}.gpkg" for c in CITIES_CONFIG]
    full_gdf = pd.concat([gpd.read_file(f) for f in gpkgs if os.path.exists(f)], ignore_index=True)

# ==========================================
# 2. CARTE 1 : MOYENNES GLOBALES (VECTEUR)
# ==========================================
print("    Generation Carte 1 : Moyennes par Quartier (Vecteur)...")

# Application de la classification
full_gdf['categorie'] = full_gdf['ndvi_moyen'].apply(reclassify_ndvi)
full_gdf['color'] = full_gdf['categorie'].map(CATEGORY_COLORS)

# Trace Grille 2x2
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

unique_cities = full_gdf['city'].unique()

for idx, city in enumerate(unique_cities):
    if idx >= len(axes): break
    ax = axes[idx]

    # Filtrer les donnees pour cette ville
    city_data = full_gdf[full_gdf['city'] == city]

    # Tracer
    city_data.plot(ax=ax, color=city_data['color'], edgecolor='black', linewidth=0.5)

    # Titres
    avg_ndvi = city_data['ndvi_moyen'].mean()
    ax.set_title(f"{city} (NDVI Moyen: {avg_ndvi:.2f})", fontsize=12, fontweight='bold')
    ax.set_axis_off()

# Legende
patches = [mpatches.Patch(color=color, label=label) for label, color in CATEGORY_COLORS.items()]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=12)

# Sauvegarde
file_vec = "output/Comparaison_Villes_Vector.png"
plt.tight_layout()
plt.subplots_adjust(bottom=0.1)
plt.savefig(file_vec, dpi=300)
plt.close()
print(f"       Sauvegarde : {file_vec}")


# ==========================================
# 3. CARTE 2 : DETAILS PIXELS (SENTINEL RASTER)
# ==========================================
print("    Generation Carte 2 : Details Pixels (Sentinel-2)...")

fig2, axes2 = plt.subplots(2, 2, figsize=(20, 16))
axes2 = axes2.flatten()

# Reutilisation de CITIES_CONFIG du Bloc 1
for idx, config in enumerate(CITIES_CONFIG):
    if idx >= len(axes2): break
    ax = axes2[idx]

    # Definir le fond en blanc explicitement
    ax.set_facecolor('white')

    city_name = config['name']
    raster_path = config['ndvi_file']
    vector_path = f"output/Result_{city_name}.gpkg" # Nous utilisons le resultat sauvegarde qui a une geometrie propre

    print(f"       Traitement {city_name}...")

    # Verification robuste : Les fichiers existent-ils ?
    if os.path.exists(raster_path) and os.path.exists(vector_path):
        try:
            # 1. Ouvrir Vecteur
            gdf_city = gpd.read_file(vector_path)

            # 2. Ouvrir Raster
            with rasterio.open(raster_path) as src:
                # 3. Aligner les Projections (CRITIQUE pour eviter l'erreur d'image)
                if gdf_city.crs != src.crs:
                    gdf_city = gdf_city.to_crs(src.crs)

                # 4. Decouper Raster (Masquage)
                # crop=True supprime le fond noir
                out_image, out_transform = mask(src, gdf_city.geometry, crop=True, nodata=np.nan)

                # 5. Extraire Donnees (Bande 1)
                ndvi_data = out_image[0]

                # Masquer les valeurs NaN pour qu'elles s'affichent en 'mauvais' (blanc)
                ndvi_masked = np.ma.masked_invalid(ndvi_data)

                # 6. Tracer avec Reclassification (Couleurs Discretes)
                # Nous utilisons l'interpolation 'nearest' pour voir les pixels exacts (aspect net)
                show(
                    ndvi_masked,
                    transform=out_transform,
                    ax=ax,
                    cmap=cmap_base,
                    norm=norm_ndvi,
                    interpolation='nearest'
                )

                # Superposer les limites des quartiers (Lignes fines pour le contexte)
                gdf_city.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.3, alpha=0.5)

                ax.set_title(f"{city_name}: Details Sentinel-2", fontsize=14, fontweight='bold')
                ax.axis('off')

        except Exception as e:
            print(f"       Erreur sur {city_name}: {e}")
            ax.text(0.5, 0.5, f"Erreur: {str(e)}", ha='center', va='center')
            ax.axis('off')
    else:
        print(f"       Fichiers manquants pour {city_name}")
        ax.text(0.5, 0.5, "Donnees Manquantes (TIF ou GPKG)", ha='center', va='center')
        ax.axis('off')

# Legende pour la Carte Pixelisee
patches_pix = [
    mpatches.Patch(color=NDVI_COLORS_LIST[0], label="Eau / Artefact (<0)"),
    mpatches.Patch(color=NDVI_COLORS_LIST[1], label="Bati / Sol nu (0-0.2)"),
    mpatches.Patch(color=NDVI_COLORS_LIST[2], label="Veg. Basse (0.2-0.4)"),
    mpatches.Patch(color=NDVI_COLORS_LIST[3], label="Veg. Moyenne (0.4-0.6)"),
    mpatches.Patch(color=NDVI_COLORS_LIST[4], label="Veg. Dense (>0.6)")
]
fig2.legend(handles=patches_pix, loc='lower center', ncol=5, fontsize=14, frameon=False)

# Sauvegarde
file_pix = "output/Comparaison_Villes_Pixel.png"
plt.tight_layout()
plt.subplots_adjust(bottom=0.05)
plt.savefig(file_pix, dpi=300)
plt.close()
print(f"      Sauvegarde : {file_pix}")

print(" GENERATION DES CARTES TERMINEE AVEC SUCCES.")

In [ ]:
import folium
import geopandas as gpd

print("--- GÉNÉRATION CARTE WEB ---")

# 1. Préparation des données
# On s'assure que full_gdf est bien un GeoDataFrame
if not isinstance(full_gdf, gpd.GeoDataFrame):
    full_gdf = gpd.GeoDataFrame(full_gdf, geometry="geometry")

# Si le CRS a sauté (ce qui arrive après un concat), on le remet (Lambert 93)
if full_gdf.crs is None:
    full_gdf.set_crs(epsg=2154, inplace=True)

# 2. Conversion GLOBALE en WGS84 (GPS)
# On convertit tout d'un coup AVANT la boucle pour éviter l'erreur "naive geometry"
try:
    web_data = full_gdf.to_crs(epsg=4326)
except Exception as e:
    print(f" Erreur de reprojection critique : {e}")
    # Fallback de secours
    web_data = full_gdf.copy()

# 3. Création de la carte
# Centrée grossièrement sur la Bretagne
m = folium.Map(location=[48.0, -2.0], zoom_start=8, tiles="CartoDB positron")

# Fonction couleur
def get_color(ndvi):
    if ndvi is None or pd.isna(ndvi): return "#ffffff" # Blanc si vide
    if ndvi < 0: return "#1f77b4"     # Eau
    if ndvi < 0.2: return "#d3d3d3"   # Béton
    if ndvi < 0.4: return "#ccff66"   # Herbe
    if ndvi < 0.6: return "#33cc33"   # Arbres
    return "#006400"                  # Forêt

# 4. Boucle d'ajout (Sur les données déjà converties)
for _, row in web_data.iterrows():
    try:
        # La géométrie est déjà en 4326, plus besoin de toucher
        geo_json =  gpd.GeoSeries([row.geometry]).__geo_interface__

        color = get_color(row['ndvi_moyen'])

        # Contenu de la bulle info
        popup_html = f"""
        <div style="font-family: Arial; width: 200px;">
            <b>{row['nom']}</b><br>
            <i>{row['city']}</i><br>
            <hr>
            NDVI: <b>{row['ndvi_moyen']:.3f}</b><br>
            Catégorie: {row['categorie']}
        </div>
        """

        folium.GeoJson(
            geo_json,
            style_function=lambda x, color=color: {
                'fillColor': color,
                'color': 'black',
                'weight': 0.5,
                'fillOpacity': 0.7
            },
            tooltip=f"{row['nom']} ({row['ndvi_moyen']:.2f})",
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(m)

    except Exception as e:
        pass # On ignore les géométries invalides ponctuelles

# Sauvegarde
output_html = "output/Carte_Interactive_Finale.html"
m.save(output_html)
print(f"Carte Web sauvegardée : {output_html}")
print("PROJET TERMINÉ AVEC SUCCÈS.")

In [ ]:
from IPython.display import display, Markdown
import graphviz

print("--- GÉNÉRATION DE LA DOCUMENTATION TECHNIQUE ---")

# 1. CRÉATION DU SCHÉMA (Flowchart)
# On utilise Graphviz pour dessiner la logique du code
dot_source = """
digraph G {
    rankdir=TB;
    node [shape=box, style="filled", fillcolor="white", fontname="Arial", fontsize=10];
    edge [fontname="Arial", fontsize=9];

    # Noeuds
    Start [label="Début du Script", shape=oval, fillcolor="#f0f0f0"];
    Config [label="1. Configuration\n(4 Villes: Nantes, Rennes, Angers, Brest)", color="#000000"];

    subgraph cluster_loop {
        label = "Boucle par Ville";
        style=dashed;
        color=grey;

        LoadGeo [label="2. Chargement Géométrie\n(GeoJSON)", color="blue", fontcolor="blue"];
        FixGeo [label="Correction & Nettoyage\n(Projection Lambert 93 + Aplatissement 3D)", color="blue", fontcolor="blue"];
        CheckCol [label="Correction Colonne Nom\n(Brest -> LIBEL)", color="red", fontcolor="red"];

        LoadImg [label="3. Chargement Image\n(Sentinel-2 NDVI)", color="green", fontcolor="green"];

        Analyze [label="4. Analyse (Masking)\nDécoupage Pixel par Pixel", shape=component];

        Result [label="Résultat: Moyenne NDVI", style="filled", fillcolor="#e6ffe6"];
    }

    Maps [label="5. Génération Cartes\n(Vecteur + Pixel)", shape=folder];
    Web [label="6. Carte Interactive\n(HTML)", shape=note];
    End [label="Fin du Traitement", shape=oval, fillcolor="#f0f0f0"];

    # Liens
    Start -> Config;
    Config -> LoadGeo;
    LoadGeo -> FixGeo;
    FixGeo -> CheckCol;
    CheckCol -> LoadImg;
    LoadImg -> Analyze;
    Analyze -> Result;
    Result -> Maps [label="Agrégation"];
    Maps -> Web;
    Web -> End;
}
"""

# 2. AFFICHAGE DU SCHÉMA
try:
    display(graphviz.Source(dot_source))
except:
    print("(Graphviz n'est pas installé, le schéma ne peut pas s'afficher)")

# 3. EXPLICATION TEXTUELLE (Format Markdown)
explication_texte = """
# Documentation du Pipeline d'Analyse

Ce script automatise l'analyse de la végétation urbaine à partir d'images satellites. Voici les étapes techniques clés :

### Étape 1 : Configuration Robuste
Nous définissons une liste de dictionnaires pour chaque ville.
* **Point Critique :** Pour Brest, nous forçons la lecture de la colonne `LIBEL` via le paramètre `col_name`. Cela permet de récupérer les vrais noms de quartiers au lieu du nom de la commune.

### Étape 2 : Moteur Géométrique (`load_city_map`)
Cette fonction télécharge les données Open Data et effectue trois nettoyages indispensables :
1. **Projection :** Tout est converti en Lambert-93 (EPSG:2154) pour correspondre au standard français.
2. **Aplatissement 3D :** Les polygones contenant des altitudes (Z) sont aplatis en 2D pour éviter les erreurs de calcul.
3. **Harmonisation :** Renommage de la colonne identifiant le quartier en `nom`.

### Étape 3 : Analyse des Pixels (`analyze_vegetation`)
Au lieu d'utiliser des statistiques zonales classiques (qui échouent souvent sur des géométries complexes), nous utilisons une approche de **Masquage (Masking)** :
* Le script "découpe" l'image satellite en utilisant la forme exacte du quartier.
* Il extrait les valeurs de chaque pixel à l'intérieur de cette forme.
* Il ignore les valeurs `NaN` (pas de données) ou `NoData` (bords noirs).
* Il calcule la moyenne arithmétique de ces pixels.

### Étape 4 : Visualisation Double
Le script génère deux types de cartes pour comparaison :
1. **Carte Vectorielle :** Affiche une couleur unique par quartier (la moyenne). Utile pour comparer les quartiers entre eux.
2. **Carte Matricielle (Pixel) :** Affiche l'image satellite réelle reclassifiée. Utile pour voir la structure interne (parcs, rues, bâtiments).

### Sorties
Tous les fichiers sont sauvegardés dans le dossier `output/` :
* `Result_Ville.gpkg` : Géométries avec les scores calculés.
* `Bilan_Complet.csv` : Tableau Excel des scores.
* `Comparaison_*.png` : Les images statiques.
* `Carte_Interactive_Finale.html` : La carte web zoomable.
"""

# Affichage du texte formaté
display(Markdown(explication_texte))